In [ ]:
from sklearn.metrics import classification_report
from collections import Counter

model.eval()

punct_preds = []
punct_true = []

cap_preds = []
cap_true = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        punct_labels = batch["punct_labels"].to(device)
        cap_labels = batch["cap_labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        punct_logits = outputs["punct_logits"]
        cap_logits = outputs["cap_logits"]

        punct_predictions = torch.argmax(punct_logits, dim=-1)
        cap_predictions = torch.argmax(cap_logits, dim=-1)

        punct_preds.extend(punct_predictions.cpu().numpy().flatten())
        punct_true.extend(punct_labels.cpu().numpy().flatten())

        cap_preds.extend(cap_predictions.cpu().numpy().flatten())
        cap_true.extend(cap_labels.cpu().numpy().flatten())


def filter_labels(preds, true):

    f_preds = []
    f_true = []

    for p, t in zip(preds, true):

        if t != -100:
            f_preds.append(p)
            f_true.append(t)

    return f_preds, f_true


punct_preds, punct_true = filter_labels(punct_preds, punct_true)
cap_preds, cap_true = filter_labels(cap_preds, cap_true)

print("\nPUNCTUATION REPORT\n")

print(
    classification_report(
        punct_true,
        punct_preds,
        labels=[0,1,2,3],
        target_names=["O", ",", ".", "?"],
        zero_division=0
    )
)

print("True labels:", Counter(punct_true))
print("Pred labels:", Counter(punct_preds))


print("\nCAPITALIZATION REPORT\n")

print(
    classification_report(
        cap_true,
        cap_preds,
        labels=[0,1,2],
        target_names=["lower", "CAP", "ALL"],
        zero_division=0
    )
)

print("True labels:", Counter(cap_true))
print("Pred labels:", Counter(cap_preds))